In [12]:
:dep rustsat = "0.7.5"
:dep rustsat-batsat = "0.7.5"
:dep batsat = "0.6.0"
:dep rand = "0.9"

// Parametres
const BATCH_MODE: bool = true;
const SEED: u64 = 42;
const N_TRIES_PAR_TAILLE: usize = 12;

println!(
    "Configuration : batch={}, seed={}, essais_par_taille={}",
    BATCH_MODE, SEED, N_TRIES_PAR_TAILLE
);

Configuration : batch=true, seed=42, essais_par_taille=12


# Frontieres-01 : Coloration d'arêtes et conjecture de Tutte (Rust)

**Navigation** : [<< App-21 VoiceLeading](../Part3-Cas-pratiques/App-21-VoiceLeading-Python.ipynb) | [Série Search](../README.md) | [Version Python](Frontieres-01-EdgeColoring-Tutte-Python.ipynb)

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :

1. **Définir** la coloration d'arêtes d'un graphe, son indice chromatique et les classes de Vizing
2. **Formuler** la 3-coloration d'arêtes d'un graphe cubique comme un CSP résolu par un solveur SAT (chaque couleur = un couplage parfait)
3. **Construire** des familles témoins : le graphe de Petersen (plus petit snark), un graphe cubique à pont, les échelles de Möbius (non planaires mais apex)
4. **Vérifier empiriquement** un théorème récent (arXiv 2608.22870, 2026) : tout graphe cubique **sans pont et apex** admet une 3-coloration d'arêtes

Ce notebook prolonge [App-2 : Coloration de Graphes](../Part3-Cas-pratiques/App-02-GraphColoring-Python.ipynb), qui traitait la coloration de **sommets** — ici, ce sont les **arêtes** que l'on colore.

C'est le portage Rust (kernel evcxr) de [Frontieres-01 en Python](Frontieres-01-EdgeColoring-Tutte-Python.ipynb) : CP-SAT y est remplacé par le solveur SAT `rustsat` (moteur `batsat`), et les outils networkx par des implémentations du carnet (ponts par parcours lowlink, planarité par le test gauche-droite de de Fraysseix–Rosenstiehl, échantillonnage par le modèle d'appariement). Les sorties numériques (temps, échantillons) sont celles du présent carnet ; la pédagogie est inchangée.

***
## 1. Introduction (~5 min)

### De la coloration de sommets à celle d'arêtes

Dans App-2, nous cherchions à colorier des **sommets** tels que deux sommets adjacents diffèrent. La **coloration d'arêtes** est le problème dual : colorier les **arêtes** d'un graphe telles que deux arêtes partageant un sommet reçoivent des couleurs distinctes. Le nombre minimal de couleurs est l'**indice chromatique** $\chi'(G)$.

Le théorème de **Vizing (1964)** encadre étroitement cet indice :

$$\Delta(G) \le \chi'(G) \le \Delta(G) + 1$$

où $\Delta(G)$ est le degré maximal. Les graphes atteignant la borne inférieure sont de **classe 1**, les autres de **classe 2**.

### Le cœur du problème : les graphes cubiques sans pont

Un graphe est **cubique** (ou 3-régulier) si tous ses sommets ont degré 3 ; il est **sans pont** si aucune arête n'est un pont (isthme). Pour un graphe cubique, $\chi'(G) \in \{3, 4\}$ : une 3-coloration d'arêtes d'un cubique revient à **partitionner ses arêtes en 3 couplages parfaits** — chaque couleur couvre chaque sommet exactement une fois.

La question « quel graphe cubique sans pont est de classe 1 ? » est structurellement difficile :

- **Conjecture de Tutte (1966)** : tout graphe cubique **planaire** sans pont est 3-arête-colorable — équivalente au théorème des quatre couleurs (résolue par conséquent en 1976).
- Un **snark** est un graphe cubique sans pont non 3-arête-colorable. Le plus petit est le **graphe de Petersen** (10 sommets).
- **Théorème (arXiv 2608.22870, 2026)** : tout graphe cubique **sans pont et apex** — un graphe dont la suppression d'**un seul sommet** le rend planaire — est 3-arête-colorable. Ce résultat étend la conjecture de Tutte au-delà du planaire, avec une preuve assistée par ordinateur et un algorithme constructif en $O(n^2)$.

Dans ce notebook, nous **vérifions empiriquement** ce théorème avec le solveur SAT. Le solveur peut fournir une coloration (statut `SAT`), prouver l'impossibilité (statut `UNSAT`), ou être interrompu par la limite de temps avant de conclure (statut indéterminé) ; ces trois situations restent explicitement distinctes.

In [13]:
// Imports pour tout le notebook
use std::time::{Duration, Instant};

use rand::prelude::*;
use rustsat::solvers::{Solve, SolverResult};
use rustsat::types::{Clause, Lit, TernaryVal, Var};
use rustsat_batsat::Solver as SolveurBatsat;

println!("rustsat 0.7.5 + batsat 0.6.0 pret");
println!("SEED = {}", SEED);

rustsat 0.7.5 + batsat 0.6.0 pret


SEED = 42


***
## 2. Définitions et familles témoins (~8 min)

Quatre propriétés structurelles vont orchestrer tout le notebook :

| Propriété | Définition | Vérification |
|---|---|---|
| **cubique** | tous les sommets ont degré 3 | parcours des degrés |
| **sans pont** | aucune arête dont la suppression déconnecte | parcours lowlink (Tarjan) |
| **planaire** | dessinable sans croisement | test gauche-droite (de Fraysseix–Rosenstiehl) |
| **apex** | il existe un sommet $v$ tel que $G - v$ soit planaire | test de planarité après suppression, pour chaque $v$ |

Remarquez l'emboîtement : **planaire $\Rightarrow$ apex** (supprimer un sommet d'un graphe planaire laisse un graphe planaire). L'intérêt du théorème de 2026 est précisément la zone **apex mais non planaire**, invisible pour Tutte-1966.

Nous assemblons nos familles témoins : $K_4$ et le prisme triangulaire (planaires triviaux), le graphe de Petersen (le snark minimal), et un graphe cubique **avec pont** construit à la main pour la section 4.

In [14]:
// Definitions structurelles
#[derive(Clone)]
struct Graphe {
    adj: Vec<Vec<usize>>,
    noms: Vec<String>,
}

impl Graphe {
    fn nouveau(noms: Vec<String>) -> Self {
        Graphe {
            adj: vec![Vec::new(); noms.len()],
            noms,
        }
    }

    fn n(&self) -> usize {
        self.adj.len()
    }

    fn m(&self) -> usize {
        self.adj.iter().map(|vois| vois.len()).sum::<usize>() / 2
    }

    fn ajoute(&mut self, u: usize, v: usize) {
        if u != v && !self.adj[u].contains(&v) {
            self.adj[u].push(v);
            self.adj[v].push(u);
        }
    }

    fn retire_arete(&mut self, u: usize, v: usize) {
        self.adj[u].retain(|&w| w != v);
        self.adj[v].retain(|&w| w != u);
    }

    fn supprime(&self, v: usize) -> Graphe {
        let mut nouveaux = vec![usize::MAX; self.n()];
        let mut noms: Vec<String> = Vec::new();
        for i in 0..self.n() {
            if i != v {
                nouveaux[i] = noms.len();
                noms.push(self.noms[i].clone());
            }
        }
        let mut h = Graphe::nouveau(noms);
        for u in 0..self.n() {
            if u == v {
                continue;
            }
            for &w in &self.adj[u] {
                if w != v {
                    h.ajoute(nouveaux[u], nouveaux[w]);
                }
            }
        }
        h
    }

    fn aretes(&self) -> Vec<(usize, usize)> {
        let mut vs = Vec::new();
        for u in 0..self.n() {
            for &v in &self.adj[u] {
                if v > u {
                    vs.push((u, v));
                }
            }
        }
        vs
    }
}

fn v(b: bool) -> &'static str {
    if b {
        "True"
    } else {
        "False"
    }
}

fn est_cubique(g: &Graphe) -> bool {
    !g.adj.is_empty() && g.adj.iter().all(|vois| vois.len() == 3)
}

fn sans_pont(g: &Graphe) -> bool {
    ponts(g).is_empty()
}

fn composantes_connexes(g: &Graphe) -> Vec<Vec<usize>> {
    let n = g.n();
    let mut vus = vec![false; n];
    let mut comps = Vec::new();
    for depart in 0..n {
        if vus[depart] {
            continue;
        }
        vus[depart] = true;
        let mut pile = vec![depart];
        let mut comp = Vec::new();
        while let Some(u) = pile.pop() {
            comp.push(u);
            for &w in &g.adj[u] {
                if !vus[w] {
                    vus[w] = true;
                    pile.push(w);
                }
            }
        }
        comp.sort();
        comps.push(comp);
    }
    comps
}

// Ponts par parcours profond lowlink (Tarjan), iteratif.
fn ponts(g: &Graphe) -> Vec<(usize, usize)> {
    let n = g.n();
    let mut visites = vec![false; n];
    let mut resultat = Vec::new();
    for depart in 0..n {
        if visites[depart] {
            continue;
        }
        visites[depart] = true;
        let mut prof = vec![usize::MAX; n];
        let mut bas = vec![usize::MAX; n];
        let mut horloge = 0usize;
        prof[depart] = 0;
        bas[depart] = 0;
        let mut pile: Vec<(usize, usize, usize)> = vec![(depart, 0, usize::MAX)];
        while let Some((u, i, pere)) = pile.last().copied() {
            if i < g.adj[u].len() {
                pile.last_mut().unwrap().1 += 1;
                let w = g.adj[u][i];
                if w == pere {
                    continue;
                }
                if visites[w] {
                    bas[u] = bas[u].min(prof[w]);
                } else {
                    horloge += 1;
                    prof[w] = horloge;
                    bas[w] = horloge;
                    visites[w] = true;
                    pile.push((w, 0, u));
                }
            } else {
                pile.pop();
                if let Some(&(parent, _, _)) = pile.last() {
                    bas[parent] = bas[parent].min(bas[u]);
                    if bas[u] > prof[parent] {
                        resultat.push((parent, u));
                    }
                }
            }
        }
    }
    resultat
}

// Test de planarite gauche-droite (de Fraysseix-Rosenstiehl), d'apres
// l'algorithme lineaire de Brandes ; portage fidele de la partie test de
// networkx.algorithms.planarity (LRPlanarity, sans construction d'embedding).
#[derive(Clone, Copy)]
struct Intervalle {
    bas: Option<usize>,
    haut: Option<usize>,
}

impl Intervalle {
    fn vide() -> Self {
        Intervalle {
            bas: None,
            haut: None,
        }
    }

    fn est_vide(&self) -> bool {
        self.bas.is_none() && self.haut.is_none()
    }
}

#[derive(Clone, Copy)]
struct PaireConflit {
    gauche: Intervalle,
    droite: Intervalle,
}

impl PaireConflit {
    fn vide() -> Self {
        PaireConflit {
            gauche: Intervalle::vide(),
            droite: Intervalle::vide(),
        }
    }

    fn echange(&mut self) {
        std::mem::swap(&mut self.gauche, &mut self.droite);
    }
}

const NON: usize = usize::MAX;

struct EtatLr<'a> {
    g: &'a Graphe,
    n: usize,
    prof: Vec<u32>,           // hauteur DFS (u32::MAX = non visite)
    bas: Vec<u32>,            // lowpt, par arete orientee u*n + w
    bas2: Vec<u32>,           // second lowpt
    nid: Vec<u32>,            // profondeur d'imbrication
    arete_parent: Vec<usize>, // id d'arete orientee (NON = racine)
    dg: Vec<Vec<usize>>,      // successeurs orientes par le DFS
    oriente: Vec<bool>,       // arete deja orientee (dans un sens ou l'autre)
    ordonne: Vec<Vec<usize>>, // successeurs tries par profondeur d'imbrication
    pile: Vec<PaireConflit>,  // pile de paires de conflit
    fond_pile: Vec<usize>,    // hauteur de pile au debut de chaque arete
    arete_bas: Vec<usize>,    // arete realisant le lowpt (NON = aucune)
    renvoi: Vec<usize>,       // chaine de references entre aretes
    cote: Vec<i32>,           // orientation relative (side), 1 par defaut
}

impl<'a> EtatLr<'a> {
    fn nouveau(g: &'a Graphe) -> Self {
        let n = g.n();
        EtatLr {
            g,
            n,
            prof: vec![u32::MAX; n],
            bas: vec![0; n * n],
            bas2: vec![0; n * n],
            nid: vec![0; n * n],
            arete_parent: vec![NON; n],
            dg: vec![Vec::new(); n],
            oriente: vec![false; n * n],
            ordonne: vec![Vec::new(); n],
            pile: Vec::new(),
            fond_pile: vec![0; n * n],
            arete_bas: vec![NON; n * n],
            renvoi: vec![NON; n * n],
            cote: vec![1; n * n],
        }
    }

    fn dfs1(&mut self, u: usize) {
        let e = self.arete_parent[u];
        for w in self.g.adj[u].clone() {
            let uw = u * self.n + w;
            let wu = w * self.n + u;
            if self.oriente[uw] || self.oriente[wu] {
                continue; // arete deja orientee
            }
            self.dg[u].push(w);
            self.oriente[uw] = true;
            self.bas[uw] = self.prof[u];
            self.bas2[uw] = self.prof[u];
            if self.prof[w] == u32::MAX {
                // arete d'arbre
                self.arete_parent[w] = uw;
                self.prof[w] = self.prof[u] + 1;
                self.dfs1(w);
            } else {
                // arete de retour
                self.bas[uw] = self.prof[w];
            }
            // profondeur d'imbrication
            self.nid[uw] = 2 * self.bas[uw];
            if self.bas2[uw] < self.prof[u] {
                self.nid[uw] += 1; // arete chordale
            }
            // mise a jour des baspoints de l'arete parente
            if e != NON {
                if self.bas[uw] < self.bas[e] {
                    self.bas2[e] = self.bas[e].min(self.bas2[uw]);
                    self.bas[e] = self.bas[uw];
                } else if self.bas[uw] > self.bas[e] {
                    self.bas2[e] = self.bas2[e].min(self.bas[uw]);
                } else {
                    self.bas2[e] = self.bas2[e].min(self.bas2[uw]);
                }
            }
        }
    }

    fn conflit(&self, iv: &Intervalle, b: usize) -> bool {
        // un intervalle entre en conflit si son sommet haut descend sous le lowpt de b
        match iv.haut {
            Some(h) => !iv.est_vide() && self.bas[h] > self.bas[b],
            None => false,
        }
    }

    fn dfs2(&mut self, u: usize) -> bool {
        let e = self.arete_parent[u];
        for w in self.ordonne[u].clone() {
            let ei = u * self.n + w;
            self.fond_pile[ei] = self.pile.len();
            if ei == self.arete_parent[w] {
                // arete d'arbre : recursion
                if !self.dfs2(w) {
                    return false;
                }
            } else {
                // arete de retour
                self.arete_bas[ei] = ei;
                self.pile.push(PaireConflit {
                    gauche: Intervalle::vide(),
                    droite: Intervalle {
                        bas: Some(ei),
                        haut: Some(ei),
                    },
                });
            }
            // integrer les nouvelles aretes de retour
            if self.bas[ei] < self.prof[u] {
                if w == self.ordonne[u][0] {
                    // e_i a une arete de retour
                    self.arete_bas[e] = self.arete_bas[ei];
                } else {
                    // ajouter les contraintes de e_i
                    if !self.ajoute_contraintes(ei, e) {
                        return false; // non planaire
                    }
                }
            }
        }
        // retirer les aretes de retour qui remontent au parent
        if e != NON {
            self.retire_aretes_retour(e);
        }
        true
    }

    fn ajoute_contraintes(&mut self, ei: usize, e: usize) -> bool {
        let mut p = PaireConflit::vide();
        // fusionner les aretes de retour de e_i dans P.droite
        loop {
            let Some(mut q) = self.pile.pop() else {
                return false; // non planaire (pile epuisee avant le fond)
            };
            if !q.gauche.est_vide() {
                q.echange();
            }
            if !q.gauche.est_vide() {
                return false; // non planaire
            }
            let Some(qbas) = q.droite.bas else {
                return false;
            };
            if self.bas[qbas] > self.bas[e] {
                // fusion d'intervalles
                if p.droite.est_vide() {
                    p.droite = q.droite; // intervalle le plus haut
                } else {
                    let (Some(phaut), Some(qhaut)) = (p.droite.haut, q.droite.haut) else {
                        return false;
                    };
                    self.renvoi[phaut] = qhaut;
                }
                p.droite.bas = q.droite.bas;
            } else {
                // alignement
                self.renvoi[qbas] = self.arete_bas[e];
            }
            if self.pile.len() == self.fond_pile[ei] {
                break;
            }
        }
        // fusionner les aretes de retour conflictuelles de e_1..e_{i-1} dans P.gauche
        loop {
            let Some(top) = self.pile.last().copied() else {
                break;
            };
            if !self.conflit(&top.gauche, ei) && !self.conflit(&top.droite, ei) {
                break;
            }
            let Some(mut q) = self.pile.pop() else {
                break;
            };
            if self.conflit(&q.droite, ei) {
                q.echange();
            }
            if self.conflit(&q.droite, ei) {
                return false; // non planaire
            }
            // fusionner l'intervalle sous le lowpt de e_i dans P.droite
            if let (Some(pbas), Some(qhaut)) = (p.droite.bas, q.droite.haut) {
                self.renvoi[pbas] = qhaut;
            }
            if q.droite.bas.is_some() {
                p.droite.bas = q.droite.bas;
            }
            // fusionner l'intervalle de Q.gauche dans P.gauche
            if p.gauche.est_vide() {
                p.gauche = q.gauche;
            } else if let (Some(pbas), Some(qhaut)) = (p.gauche.bas, q.gauche.haut) {
                self.renvoi[pbas] = qhaut;
            }
            p.gauche.bas = q.gauche.bas;
        }
        if !(p.gauche.est_vide() && p.droite.est_vide()) {
            self.pile.push(p);
        }
        true
    }

    fn retire_aretes_retour(&mut self, e: usize) {
        let u = e / self.n;
        // retirer les aretes de retour qui remontent au parent u
        // (supprimer des paires de conflit entieres)
        while let Some(top) = self.pile.last().copied() {
            let plus_bas = |iv: &Intervalle| -> u32 {
                iv.bas.map(|x| self.bas[x]).unwrap_or(u32::MAX)
            };
            let plus_bas_paire = if top.gauche.est_vide() {
                plus_bas(&top.droite)
            } else if top.droite.est_vide() {
                plus_bas(&top.gauche)
            } else {
                plus_bas(&top.gauche).min(plus_bas(&top.droite))
            };
            if plus_bas_paire == self.prof[u] {
                if let Some(p) = self.pile.pop() {
                    if let Some(x) = p.gauche.bas {
                        self.cote[x] = -1;
                    }
                }
            } else {
                break;
            }
        }
        // une paire de conflit de plus a considerer
        if let Some(mut p) = self.pile.pop() {
            // rogner l'intervalle gauche
            while let Some(h) = p.gauche.haut {
                if h % self.n == u {
                    let nxt = self.renvoi[h];
                    p.gauche.haut = if nxt == NON { None } else { Some(nxt) };
                } else {
                    break;
                }
            }
            if p.gauche.haut.is_none() && p.gauche.bas.is_some() {
                if let Some(x) = p.gauche.bas {
                    self.renvoi[x] = p.droite.bas.unwrap_or(NON);
                    self.cote[x] = -1;
                }
                p.gauche.bas = None;
            }
            // rogner l'intervalle droit
            while let Some(h) = p.droite.haut {
                if h % self.n == u {
                    let nxt = self.renvoi[h];
                    p.droite.haut = if nxt == NON { None } else { Some(nxt) };
                } else {
                    break;
                }
            }
            if p.droite.haut.is_none() && p.droite.bas.is_some() {
                if let Some(x) = p.droite.bas {
                    self.renvoi[x] = p.gauche.bas.unwrap_or(NON);
                    self.cote[x] = -1;
                }
                p.droite.bas = None;
            }
            self.pile.push(p);
        }
        // le cote de e est le cote d'une arete de retour la plus haute
        if self.bas[e] < self.prof[u] {
            if let Some(top) = self.pile.last().copied() {
                let hl = top.gauche.haut;
                let hr = top.droite.haut;
                self.renvoi[e] = match (hl, hr) {
                    (Some(h), None) => h,
                    (Some(h), Some(r)) if self.bas[h] > self.bas[r] => h,
                    (_, Some(r)) => r,
                    (None, None) => NON,
                };
            }
        }
    }
}

fn est_planaire(g: &Graphe) -> bool {
    let n = g.n();
    if n <= 2 {
        return true;
    }
    if g.m() > 3 * n - 6 {
        return false;
    }
    let mut etat = EtatLr::nouveau(g);
    for u in 0..n {
        if etat.prof[u] == u32::MAX {
            etat.prof[u] = 0;
            etat.dfs1(u);
        }
    }
    for u in 0..n {
        let mut ordre = etat.dg[u].clone();
        ordre.sort_by_key(|&w| etat.nid[u * n + w]);
        etat.ordonne[u] = ordre;
    }
    for u in 0..n {
        if etat.arete_parent[u] == NON {
            if !etat.dfs2(u) {
                return false;
            }
        }
    }
    true
}

fn est_apex(g: &Graphe) -> bool {
    for v in 0..g.n() {
        if est_planaire(&g.supprime(v)) {
            return true;
        }
    }
    false
}

#[derive(Clone, Debug)]
struct Description {
    graphe: String,
    n: usize,
    m: usize,
    cubique: bool,
    sans_pont: bool,
    planaire: bool,
    apex: bool,
}

fn decrire(nom: &str, g: &Graphe) -> Description {
    Description {
        graphe: nom.to_string(),
        n: g.n(),
        m: g.m(),
        cubique: est_cubique(g),
        sans_pont: sans_pont(g),
        planaire: est_planaire(g),
        apex: est_apex(g),
    }
}

fn table_temoins(descs: &[Description]) {
    println!(
        "{:<22}{:>3}{:>4}  {:<8}{:<10}{:<9}{}",
        "graphe", "n", "m", "cubique", "sans_pont", "planaire", "apex"
    );
    for d in descs {
        println!(
            "{:<22}{:>3}{:>4}  {:<8}{:<10}{:<9}{}",
            d.graphe,
            d.n,
            d.m,
            v(d.cubique),
            v(d.sans_pont),
            v(d.planaire),
            v(d.apex)
        );
    }
}

// Constructeurs de graphes
fn graphe_complet(k: usize) -> Graphe {
    let noms = (0..k).map(|i| i.to_string()).collect();
    let mut g = Graphe::nouveau(noms);
    for i in 0..k {
        for j in (i + 1)..k {
            g.ajoute(i, j);
        }
    }
    g
}

fn prisme_triangulaire() -> Graphe {
    let noms = (0..6).map(|i| i.to_string()).collect();
    let mut g = Graphe::nouveau(noms);
    for i in 0..3 {
        g.ajoute(i, (i + 1) % 3);
        g.ajoute(i + 3, (i + 1) % 3 + 3);
        g.ajoute(i, i + 3);
    }
    g
}

fn petersen() -> Graphe {
    let noms = (0..10).map(|i| i.to_string()).collect();
    let mut g = Graphe::nouveau(noms);
    for i in 0..5 {
        g.ajoute(i, (i + 1) % 5);
        g.ajoute(i + 5, (i + 2) % 5 + 5);
        g.ajoute(i, i + 5);
    }
    g
}

fn graphe_cubique_a_pont() -> Graphe {
    // Deux copies de K4 moins une arete, recollees par un sommet de chaque
    // cote, relies par un unique pont. Cubique, avec exactement un pont.
    let mut noms: Vec<String> = Vec::new();
    for tag in ["a", "b"] {
        for i in 0..4 {
            noms.push(format!("{}{}", tag, i));
        }
    }
    noms.push("wa".to_string());
    noms.push("wb".to_string());
    let mut g = Graphe::nouveau(noms);
    for tag in [0usize, 4usize] {
        for i in 0..4 {
            for j in (i + 1)..4 {
                if (i, j) != (2, 3) {
                    // on retire l'arete (2,3) de chaque K4
                    g.ajoute(tag + i, tag + j);
                }
            }
        }
    }
    let wa = 8;
    let wb = 9;
    g.ajoute(wa, 2); // wa -- a2
    g.ajoute(wa, 3); // wa -- a3
    g.ajoute(wb, 6); // wb -- b2
    g.ajoute(wb, 7); // wb -- b3
    g.ajoute(wa, wb); // le pont
    g
}

let TEMOINS: Vec<(&str, Graphe)> = vec![
    ("K4", graphe_complet(4)),
    ("prisme triangulaire", prisme_triangulaire()),
    ("Petersen", petersen()),
    ("cubique a pont", graphe_cubique_a_pont()),
];

let descs: Vec<Description> = TEMOINS
    .iter()
    .map(|(nom, g)| decrire(nom, g))
    .collect();
table_temoins(&descs);

graphe                  n   m  cubique sans_pont planaire apex


K4                      4   6  True    True      True     True


prisme triangulaire     6   9  True    True      True     True


Petersen               10  15  True    True      False    False


cubique a pont         10  15  True    False     True     True


### Interprétation : lecture du tableau

Le tableau confirme l'orthogonalité des propriétés :

- **K4** et le **prisme** : planaires et donc apex — la 3-coloration est garantie par Tutte-1966 (via le théorème des quatre couleurs).
- **Petersen** : cubique, sans pont, mais **non planaire et non apex**. Il ne *peut pas* être apex si le théorème de 2026 est vrai — sinon il serait 3-colorable, ce qu'il n'est pas (section 3). L'expérience confirmera ce que la logique impose.
- Le **cubique à pont** : planaire, mais son pont va précisément lui interdire toute 3-coloration (section 4) — l'hypothèse « sans pont » n'est pas décorative.

In [15]:
// Visualisation textuelle : Petersen et le graphe a pont
let PETERSEN = TEMOINS[2].1.clone();
let G_PONT = TEMOINS[3].1.clone();

fn affiche_adjacences(g: &Graphe, titre: &str, marque: Option<(usize, usize)>) {
    println!("{}", titre);
    for u in 0..g.n() {
        let voisins: Vec<String> = g.adj[u]
            .iter()
            .map(|&w| {
                let est_marque = marque == Some((u, w)) || marque == Some((w, u));
                if est_marque {
                    format!("[{}]", g.noms[w])
                } else {
                    g.noms[w].clone()
                }
            })
            .collect();
        println!("  {:>4} : {}", g.noms[u], voisins.join(" "));
    }
}

affiche_adjacences(&PETERSEN, "Graphe de Petersen (10 sommets, 15 aretes) :", None);

if let Some(&(pa, pb)) = ponts(&G_PONT).first() {
    affiche_adjacences(&G_PONT, "Cubique a pont -- le pont entre crochets :", Some((pa, pb)));
}

Graphe de Petersen (10 sommets, 15 aretes) :


     0 : 1 5 4


     1 : 0 2 6


     2 : 1 3 7


     3 : 2 4 8


     4 : 3 0 9


... (13 outputs left out)

    b3 : b0 b1 wb


    wa : a2 a3 [wb]


    wb : b2 b3 [wa]


()

### Interprétation visuelle

Sur le Petersen ci-dessus : sa symétrie pentagonale le rend non planaire (il contient une subdivision de $K_{3,3}$). Sur le graphe à pont : les deux copies de $K_4$ amputées d'une arête sont recollées par l'arête entre crochets — le **pont** : le seul chemin entre les deux moitiés. Nous verrons en section 4 pourquoi cette seule arête rend la 3-coloration impossible, quand bien même le graphe est planaire.

***
## 3. Formulation SAT de la k-coloration d'arêtes (~8 min)

La traduction en CSP est directe et c'est là toute sa force :

- **Variables** : $x_{e,c} \in \{0, 1\}$ pour chaque arête $e$ et couleur $c \in \{0, \dots, k-1\}$ — « l'arête $e$ reçoit-elle la couleur $c$ ? »
- **Contrainte 1** (une couleur par arête) : pour chaque arête $e$, $\sum_c x_{e,c} = 1$
- **Contrainte 2** (pas de conflit au sommet) : pour chaque sommet $v$ et chaque couleur $c$, **au plus une** arête incidente à $v$ porte la couleur $c$

Pour un graphe cubique avec $k = 3$, la contrainte 2 devient « exactement une » par principe des tiroirs (3 arêtes incidentes, 3 couleurs distinctes) : chaque couleur forme alors un **couplage parfait** — $n$ sommets, $n/2$ arêtes par couleur, $3n/2$ arêtes au total, la partition exacte recherchée. Avec $k = 4$ (borne de Vizing), une couleur peut manquer en chaque sommet.

Le solveur SAT distingue trois états : `SAT` fournit une coloration ; `UNSAT` prouve qu'aucune coloration n'existe ; un statut indéterminé (interruption par la limite de temps) ne dit rien. **Seul `UNSAT` est un certificat d'impossibilité.**

In [16]:
// Modele SAT de la k-coloration d'aretes
use std::sync::atomic::{AtomicU64, Ordering};

// Le solveur batsat ne recoit son callback qu'a la construction, et
// rustsat_batsat::Solver n'expose qu'un Default : l'echeance de temps passe
// donc par un compteur global consulte par le callback a chaque iteration.
static ECHEANCE_MS: AtomicU64 = AtomicU64::new(u64::MAX);

fn maintenant_ms() -> u64 {
    std::time::SystemTime::now()
        .duration_since(std::time::UNIX_EPOCH)
        .map(|d| d.as_millis() as u64)
        .unwrap_or(0)
}

struct LimiteTemps;

impl Default for LimiteTemps {
    fn default() -> Self {
        LimiteTemps
    }
}

impl batsat::callbacks::Callbacks for LimiteTemps {
    fn stop(&self) -> bool {
        let e = ECHEANCE_MS.load(Ordering::Relaxed);
        e != u64::MAX && maintenant_ms() >= e
    }
}

type SolveurLimite = SolveurBatsat<LimiteTemps>;

fn coloration_aretes_sat(g: &Graphe, k: usize, limite_ms: u64) -> (Option<Vec<usize>>, String) {
    // Resout la k-coloration d'aretes et preserve le statut du solveur.
    // Retourne (coloration, statut) : la coloration existe pour "SAT",
    // elle est None pour "UNSAT" (impossibilite certifiee) ou "UNKNOWN"
    // (limite de temps atteinte avant conclusion).
    let aretes = g.aretes();
    let na = aretes.len();
    let mut solveur = SolveurLimite::default();

    if limite_ms == u64::MAX {
        ECHEANCE_MS.store(u64::MAX, Ordering::Relaxed);
    } else {
        ECHEANCE_MS.store(maintenant_ms() + limite_ms, Ordering::Relaxed);
    }

    let x = |e: usize, c: usize| Lit::new((e * k + c) as u32, false);
    if na * k > 0 {
        solveur
            .reserve(Var::new((na * k - 1) as u32))
            .expect("reservation des variables");
    }

    // Contrainte 1 : exactement une couleur par arete (clause ALO + AMO)
    for e in 0..na {
        let toutes: Vec<Lit> = (0..k).map(|c| x(e, c)).collect();
        solveur
            .add_clause_ref(&Clause::from(&toutes[..]))
            .expect("clause alo");
        for c1 in 0..k {
            for c2 in (c1 + 1)..k {
                solveur
                    .add_clause_ref(&Clause::from([!x(e, c1), !x(e, c2)]))
                    .expect("clause amo arete");
            }
        }
    }

    // Contrainte 2 : au plus une arete de couleur c incidente a chaque sommet.
    // Pour k = degre, le principe des tiroirs force exactement une
    // (couplage parfait) ; pour k = degre + 1 (borne de Vizing), une couleur
    // peut manquer.
    for v in 0..g.n() {
        let incidentes: Vec<usize> = (0..na)
            .filter(|&i| aretes[i].0 == v || aretes[i].1 == v)
            .collect();
        for c in 0..k {
            for a in 0..incidentes.len() {
                for b in (a + 1)..incidentes.len() {
                    solveur
                        .add_clause_ref(&Clause::from([
                            !x(incidentes[a], c),
                            !x(incidentes[b], c),
                        ]))
                        .expect("clause conflit sommet");
                }
            }
        }
    }

    match solveur.solve() {
        Ok(SolverResult::Sat) => {
            let mut coloration = vec![0usize; na];
            for e in 0..na {
                for c in 0..k {
                    if matches!(solveur.lit_val(x(e, c)), Ok(TernaryVal::True)) {
                        coloration[e] = c;
                    }
                }
            }
            (Some(coloration), "SAT".to_string())
        }
        Ok(SolverResult::Unsat) => (None, "UNSAT".to_string()),
        _ => (None, "UNKNOWN".to_string()),
    }
}

fn verdict_texte(coloration: &Option<Vec<usize>>, statut: &str) -> String {
    if coloration.is_some() {
        "3-colorable".to_string()
    } else if statut == "UNSAT" {
        "INFEASIBLE (chi' >= 4)".to_string()
    } else {
        format!("INDETERMINE ({})", statut)
    }
}

// Verification sur les temoins, k = 3
let mut verdicts: Vec<(String, bool, f64)> = Vec::new();
for (nom, g) in TEMOINS.iter() {
    let t0 = Instant::now();
    let (coloration, statut) = coloration_aretes_sat(g, 3, 10_000);
    let dt_ms = t0.elapsed().as_secs_f64() * 1000.0;
    verdicts.push((nom.to_string(), coloration.is_some(), dt_ms));
    let etat = verdict_texte(&coloration, &statut);
    println!("{:<22} -> {:<28} ({:.1} ms)", nom, etat, dt_ms);
}

// Et la 4-coloration de Petersen (borne de Vizing atteinte) ?
let t0 = Instant::now();
let (col4, statut4) = coloration_aretes_sat(&PETERSEN, 4, 10_000);
let etat4 = if col4.is_some() {
    "trouve".to_string()
} else {
    format!("non trouve ({})", statut4)
};
println!(
    "Petersen en 4 couleurs -> {} ({:.1} ms)",
    etat4,
    t0.elapsed().as_secs_f64() * 1000.0
);

// Controle negatif : une limite quasi nulle ne prouve PAS l'impossibilite.
let (col_timeout, statut_timeout) = coloration_aretes_sat(&PETERSEN, 3, 0);
println!(
    "Petersen, limite 0 ms -> {}",
    verdict_texte(&col_timeout, &statut_timeout)
);
assert_eq!(statut_timeout, "UNKNOWN");

K4                     -> 3-colorable                  (0.1 ms)


prisme triangulaire    -> 3-colorable                  (0.0 ms)


Petersen               -> INFEASIBLE (chi' >= 4)       (0.2 ms)


cubique a pont         -> INFEASIBLE (chi' >= 4)       (0.1 ms)


Petersen en 4 couleurs -> trouve (0.1 ms)


Petersen, limite 0 ms -> INDETERMINE (UNKNOWN)


### Interprétation : le solveur discrimine

Le verdict est net, et c'est un contraste **structurel** (pas une différence de vitesse) :

- **K4, prisme** : 3-colorables, comme l'exige Tutte-1966 pour les planaires sans pont.
- **Petersen** : `INFEASIBLE` en $k=3$ — le solveur a **épuisé l'espace de recherche** et prouvé qu'aucune partition en 3 couplages parfaits n'existe. Avec $k=4$, une coloration est trouvée : $\chi'(\text{Petersen}) = 4 = \Delta + 1$, il est de **classe 2** — c'est la définition d'un snark.
- Le **cubique à pont** : `INFEASIBLE` en $k=3$ alors qu'il est planaire — première alerte que l'hypothèse « sans pont » est indispensable (section 4).

Notons la différence de nature entre les deux sorties : SAT donne un **témoin** (la coloration), UNSAT donne un **certificat** (aucune n'existe). Le même instrument prouve les deux.

**Ce que Rust apporte sur les mêmes instances** : la colonne de temps se confronte au jumeau Python — K4 passe de 49,2 ms (Python) à 0,1 ms ici, le prisme de 14,1 ms à 0,0 ms, Petersen de 2,0 ms à 0,2 ms et sa 4-coloration de 28,7 ms à 0,1 ms. L'écart décroît à mesure que l'instance durcit (×500 sur K4, ×10 sur Petersen) : la part dominante du temps Python est un **coût fixe par appel** (construction du modèle, enveloppe Python), que Rust élimine ; le temps de résolution lui-même est déjà petit sur ces instances jouet. À cette échelle, aucune des deux colonnes ne se voit à l'œil nu — l'écart **compte dès que les appels se multiplient**, comme l'échantillonnage de 48 cubiques aléatoires de la section 5, où chaque instance repaye le coût fixe côté Python. Ce que Rust ne change pas : le verdict (`3-colorable` / `INFEASIBLE`), le nombre de couleurs ($\chi' = 3$ ou $4$) et la structure de la recherche — l'argument du tableau reste structurel, la vitesse est un bonus d'outillage.

In [18]:
// Affichage textuel des colorations trouvees
fn affiche_coloration(g: &Graphe, titre: &str, k: usize, coloration: &Option<Vec<usize>>) {
    let aretes = g.aretes();
    let col = coloration
        .as_ref()
        .expect("coloration attendue pour l'affichage");
    println!("{}", titre);
    for c in 0..k {
        let aretes_c: Vec<String> = aretes
            .iter()
            .enumerate()
            .filter(|&(i, _)| col[i] == c)
            .map(|(_, &(a, b))| format!("{}-{}", g.noms[a], g.noms[b]))
            .collect();
        println!("  couleur {} : {}", c, aretes_c.join(", "));
    }
}

let prisme = TEMOINS[1].1.clone();
let (col_p, statut_p) = coloration_aretes_sat(&prisme, 3, 10_000);
assert!(
    col_p.is_some(),
    "coloration du prisme non trouvee : {}",
    statut_p
);
affiche_coloration(&prisme, "Prisme : 3-coloration d'aretes (chi' = 3) :", 3, &col_p);

assert!(
    col4.is_some(),
    "4-coloration de Petersen non trouvee : {}",
    statut4
);
affiche_coloration(
    &PETERSEN,
    "Petersen : il faut 4 couleurs (chi' = 4) -- snark :",
    4,
    &col4,
);

// Chaque couleur du prisme forme-t-elle bien un couplage parfait ?
let aretes_prisme = prisme.aretes();
for c in 0..3 {
    let sommets_c: Vec<usize> = {
        let mut s: Vec<usize> = col_p
        .as_ref()
        .unwrap()
        .iter()
        .enumerate()
        .filter(|&(_, &col)| col == c)
        .flat_map(|(i, _)| [aretes_prisme[i].0, aretes_prisme[i].1])
        .collect();
        s.sort();
        s
    };
    println!(
        "prisme, couleur {} : {} aretes couvrant {:?}",
        c,
        sommets_c.len() / 2,
        sommets_c
    );
    assert_eq!(sommets_c, (0..6).collect::<Vec<usize>>());
}

Prisme : 3-coloration d'aretes (chi' = 3) :


  couleur 0 : 0-3, 1-2, 4-5


  couleur 1 : 0-2, 1-4, 3-5


  couleur 2 : 0-1, 2-5, 3-4


Petersen : il faut 4 couleurs (chi' = 4) -- snark :


  couleur 0 : 0-5, 1-2, 3-4, 6-8, 7-9


... (3 outputs left out)

prisme, couleur 0 : 3 aretes couvrant [0, 1, 2, 3, 4, 5]


prisme, couleur 1 : 3 aretes couvrant [0, 1, 2, 3, 4, 5]


prisme, couleur 2 : 3 aretes couvrant [0, 1, 2, 3, 4, 5]


()

### Interprétation : chaque couleur est un couplage parfait

Sur le prisme, le décompte par couleur confirme la structure : chaque couleur couvre les **6 sommets** en **3 arêtes** disjointes — un couplage parfait. Les trois couleurs partitionnent les 9 arêtes : $3 \times 3 = 9$. Sur Petersen, la 4ᵉ couleur est inévitable. Avec quatre couleurs, le principe des tiroirs ne force plus chaque classe à couvrir tous les sommets ; une classe peut néanmoins être un couplage parfait selon la coloration trouvée.

***
## 4. Pourquoi « sans pont » est nécessaire (~5 min)

L'argument est un classique de parité. Supposons une 3-coloration d'arêtes d'un graphe cubique $G$ contenant un pont $e = xy$, et notons $A$ la composante de $G - e$ contenant $x$. Alors :

1. Chaque couleur forme un **couplage parfait** de $G$. Choisissons l'une des **deux couleurs qui ne colorent pas le pont** : aucune arête de cette couleur ne sort de $A$, puisque le pont est la seule arête de la coupe.
2. Cette couleur doit donc couvrir tous les sommets de $A$ par des arêtes internes, deux par deux : $|A|$ doit être **pair**.
3. Mais en sommant les degrés dans $A$ : $3|A| = 2|E_A| + 1$ (le pont compte pour 1), donc $|A|$ est **impair**.

Contradiction : **aucun graphe cubique avec un pont n'est 3-arête-colorable**. Le pont est un obstacle catégoriel, pas un accident. C'est pourquoi toutes les conjectures de cette famille (Tutte 1966, le théorème apex de 2026) portent sur les graphes **sans pont**.

In [20]:
// Verifications sur le graphe a pont
let liste_ponts = ponts(&G_PONT);
let noms_ponts: Vec<String> = liste_ponts
    .iter()
    .map(|&(a, b)| format!("('{}', '{}')", G_PONT.noms[a], G_PONT.noms[b]))
    .collect();
println!("ponts : [{}]", noms_ponts.join(", "));

// Decomposition par rapport au pont wa-wb : les composantes de G - pont
if let Some(&(pa, pb)) = liste_ponts.first() {
    let mut h = G_PONT.clone();
    h.retire_arete(pa, pb);
    for (i, comp) in composantes_connexes(&h).iter().enumerate() {
        let nb_aretes: usize = comp
            .iter()
            .map(|&u| G_PONT.adj[u].iter().filter(|&&w| comp.contains(&w)).count())
            .sum::<usize>()
            / 2;
        println!(
            "composante {} : |A| = {} sommets (parite : {}), 3|A| = {}, 2|E_A| + 1 = {}",
            i,
            comp.len(),
            if comp.len() % 2 == 0 { "pair" } else { "IMPAIR" },
            3 * comp.len(),
            2 * nb_aretes + 1
        );
    }
}

let (col3_pont, statut_pont) = coloration_aretes_sat(&G_PONT, 3, 10_000);
println!(
    "3-coloration du graphe a pont : {}",
    verdict_texte(&col3_pont, &statut_pont)
);

ponts : [('wa', 'wb')]


composante 0 : |A| = 5 sommets (parite : IMPAIR), 3|A| = 15, 2|E_A| + 1 = 15


composante 1 : |A| = 5 sommets (parite : IMPAIR), 3|A| = 15, 2|E_A| + 1 = 15


3-coloration du graphe a pont : INFEASIBLE (chi' >= 4)


### Interprétation : la parité à l'œuvre

Les composantes de $G - \text{pont}$ ont un nombre de sommets **impair**, et la relation $3|A| = 2|E_A| + 1$ se lit directement dans la sortie : c'est exactement le point 3 de l'argument. Le solveur confirme : `INFEASIBLE` en $k=3$ — le solveur et l'argument de parité racontent la même histoire, l'un par énumération exhaustive, l'autre en trois lignes. La preuve mathématique est plus satisfaisante ; le solveur la **vérifie** et généralise le contrôle à n'importe quel graphe.

***
## 5. Graphes apex non planaires : le théorème de 2026 à l'épreuve (~10 min)

Tutte-1966 couvre le planaire ; le théorème de arXiv 2608.22870 (2026) étend la 3-coloration aux cubiques sans pont **apex** — y compris celles qui sont **non planaires**, zone où aucune conjecture antérieure n'osait s'aventurer.

Une famille naturelle habite cette zone : les **échelles de Möbius** $M_n$. On les construit en ajoutant à un cycle de $n$ sommets les « barreaux » reliant chaque sommet à son opposé ($i \leftrightarrow i + n/2$). Pour $n \ge 8$ elles sont non planaires ; nous vérifions qu'elles restent **apex** (supprimer un sommet bien choisi dénoue le ruban) et **3-colorables**.

Puis nous lançons l'échantillonnage : des cubiques aléatoires sans pont de tailles croissantes, chacune classée (planaire / apex / autre) et soumise au solveur SAT. **Prédiction du théorème : aucune instance apex ne doit ressortir INFEASIBLE.**

In [21]:
// Echelles de Mobius : non planaires, apex, 3-colorables
fn echelle_mobius(n: usize) -> Graphe {
    // Echelle de Mobius M_n : cycle de n sommets + barreaux i -- (i + n/2).
    let noms = (0..n).map(|i| i.to_string()).collect();
    let mut g = Graphe::nouveau(noms);
    for i in 0..n {
        g.ajoute(i, (i + 1) % n);
    }
    for i in 0..n / 2 {
        g.ajoute(i, (i + n / 2) % n);
    }
    g
}

let MOBIUS: Vec<(String, Graphe)> = [8usize, 10, 12]
    .iter()
    .map(|&n| (format!("M{}", n), echelle_mobius(n)))
    .collect();
for (nom, g) in MOBIUS.iter() {
    let t0 = Instant::now();
    let (coloration, statut) = coloration_aretes_sat(g, 3, 10_000);
    let dt = t0.elapsed().as_secs_f64() * 1000.0;
    let verdict = if coloration.is_some() {
        "oui".to_string()
    } else {
        format!("non ({})", statut)
    };
    println!("{:<5} 3-colorable = {} ({:.1} ms)", nom, verdict, dt);
}

let descs_mobius: Vec<Description> = MOBIUS
    .iter()
    .map(|(nom, g)| decrire(nom, g))
    .collect();
table_temoins(&descs_mobius);

M8    3-colorable = oui (0.1 ms)


M10   3-colorable = oui (0.0 ms)


M12   3-colorable = oui (0.0 ms)


graphe                  n   m  cubique sans_pont planaire apex


M8                      8  12  True    True      False    True


M10                    10  15  True    True      False    True


M12                    12  18  True    True      False    True


### Interprétation : la zone apex-non-planaire existe et est habitable

Le tableau des échelles de Möbius montre la configuration clé : **`planaire = False` et `apex = True` simultanément**, avec la 3-coloration trouvée à chaque fois. Ces graphes sont **hors de portée** de Tutte-1966 (non planaires) mais **couverts** par le théorème apex de 2026 — et le solveur confirme. La frontière du théorème n'est pas le plan : c'est la suppressibilité d'un unique sommet.

In [25]:
// Echantillonnage : cubiques aleatoires sans pont, classifiees et testees
use std::collections::BTreeMap;

// Une variable persistee `v` de la session masque le fn v(bool) de la cellule
// d'outillage : on la remplace par un pointeur de fonction equivalent.
let v: fn(bool) -> &'static str = |b| if b { "True" } else { "False" };

fn graphe_regulier_aleatoire(d: usize, n: usize, rng: &mut StdRng) -> Graphe {
    // Modele d'appariement (pairing model), portage du generateur
    // networkx.random_regular_graph.
    let contient = |aretes: &[(usize, usize)], s1: usize, s2: usize| {
        aretes.iter().any(|&(a, b)| a == s1 && b == s2)
    };
    'tentative: loop {
        let mut aretes: Vec<(usize, usize)> = Vec::new();
        let mut stubs: Vec<usize> = (0..n)
            .flat_map(|sommet| std::iter::repeat(sommet).take(d))
            .collect();
        while !stubs.is_empty() {
            stubs.shuffle(rng);
            let mut potentiels: BTreeMap<usize, usize> = BTreeMap::new();
            let mut i = 0;
            while i + 1 < stubs.len() {
                let (mut s1, mut s2) = (stubs[i], stubs[i + 1]);
                if s1 > s2 {
                    std::mem::swap(&mut s1, &mut s2);
                }
                if s1 != s2 && !contient(&aretes, s1, s2) {
                    aretes.push((s1, s2));
                } else {
                    *potentiels.entry(s1).or_insert(0) += 1;
                    *potentiels.entry(s2).or_insert(0) += 1;
                }
                i += 2;
            }
            // convient ? une paire de sommets distincts encore disponibles
            let cles: Vec<usize> = potentiels.keys().copied().collect();
            let convient = if cles.is_empty() {
                true
            } else {
                (0..cles.len()).any(|t| {
                    (0..t).any(|u2| {
                        let (lo, hi) = if cles[t] < cles[u2] {
                            (cles[t], cles[u2])
                        } else {
                            (cles[u2], cles[t])
                        };
                        !contient(&aretes, lo, hi)
                    })
                })
            };
            if !convient {
                continue 'tentative;
            }
            stubs = potentiels
                .iter()
                .flat_map(|(&sommet, &occ)| std::iter::repeat(sommet).take(occ))
                .collect();
        }
        // reussite : construction du graphe
        let noms = (0..n).map(|i| i.to_string()).collect();
        let mut g = Graphe::nouveau(noms);
        for &(a, b) in &aretes {
            g.ajoute(a, b);
        }
        return g;
    }
}

let mut rng = StdRng::seed_from_u64(SEED);
let mut lignes: Vec<(usize, bool, bool, String, bool, f64)> = Vec::new();
let mut n_apex = 0usize;
let mut n_non_apex = 0usize;
let mut n_snark = 0usize;
let mut n_indetermine = 0usize;

for n in [10usize, 12, 14, 16] {
    let mut trouves = 0usize;
    let mut essais = 0usize;
    while trouves < N_TRIES_PAR_TAILLE && essais < 200 {
        essais += 1;
        let graine = rng.next_u32() as u64;
        let mut rng_g = StdRng::seed_from_u64(graine);
        let g = graphe_regulier_aleatoire(3, n, &mut rng_g);
        if !sans_pont(&g) {
            continue;
        }
        trouves += 1;
        let t0 = Instant::now();
        let (col, statut) = coloration_aretes_sat(&g, 3, 10_000);
        let dt = t0.elapsed().as_secs_f64() * 1000.0;
        let ap = est_apex(&g);
        let pl = est_planaire(&g);
        let colore = col.is_some();
        let infeasible = statut == "UNSAT";
        lignes.push((n, pl, ap, statut.clone(), colore, (dt * 10.0).round() / 10.0));
        n_apex += ap as usize;
        n_non_apex += (!ap) as usize;
        n_snark += infeasible as usize;
        n_indetermine += (statut == "UNKNOWN") as usize;
    }
}

println!("instances sans pont echantillonnees : {}", lignes.len());
println!("  apex       : {}", n_apex);
println!("  non apex   : {}", n_non_apex);
println!("  snarks (INFEASIBLE) : {}", n_snark);
println!("  indetermines        : {}", n_indetermine);
println!(
    "  apex ET INFEASIBLE  : {}",
    lignes.iter().filter(|l| l.2 && l.3 == "UNSAT").count()
);

// Regroupement (equivalent du groupby pandas)
let mut comptage: BTreeMap<(usize, bool, bool, String, bool), usize> = BTreeMap::new();
for l in &lignes {
    *comptage
        .entry((l.0, l.1, l.2, l.3.clone(), l.4))
        .or_insert(0) += 1;
}
println!(
    "{:>3}  {:>9}  {:>5}  {:>8}  {:>11}  nb",
    "n", "planaire", "apex", "statut", "3-colorable"
);
for ((n, pl, ap, statut, c3), nb) in &comptage {
    println!(
        "{:>3}  {:>9}  {:>5}  {:>8}  {:>11}  {}",
        n,
        v(*pl),
        v(*ap),
        statut,
        v(*c3),
        nb
    );
}

instances sans pont echantillonnees : 48


  apex       : 46


  non apex   : 2


  snarks (INFEASIBLE) : 0


  indetermines        : 0


  apex ET INFEASIBLE  : 0


... (8 outputs left out)

 16      False  False       SAT         True  1


 16      False   True       SAT         True  6


 16       True   True       SAT         True  5


()

### Interprétation : le théorème tient sur chaque instance résolue

Le décompte final est le message du notebook :

- Toutes les instances **apex** échantillonnées — planaires ou non — ont été résolues `SAT` et sont ressorties **3-colorables** : aucune violation du théorème de 2026.
- Aucun **snark** (`INFEASIBLE`) n'est apparu dans l'échantillon aléatoire : ils sont rares (le Petersen est l'unique snark à 10 sommets, et la probabilité de le tirer au hasard est faible) — c'est cohérent avec leur réputation de « pathologies structurelles » plutôt que de graphes génériques.
- Le compteur `indetermines` rend visibles les statuts `UNKNOWN` : ils ne seraient jamais assimilés à une preuve d'impossibilité.
- À ces petites tailles, la propriété apex est fréquente : retirer un sommet à un graphe cubique de 10 à 16 sommets laisse assez peu d'arêtes pour que la planarité se rétablisse souvent.

**Limite d'honnêteté méthodologique** : cette vérification est *empirique et énumérative* — elle conforte, elle ne démontre pas. La preuve du théorème (réductibilité + déchargement assistés par ordinateur, algorithme constructif $O(n^2)$) vit dans le papier. Notons enfin que le papier est une **v1 d'août 2026, récente et non encore revue** ; les reconstructions indépendantes citées par les auteurs valident la reproductibilité des *calculs*, pas la correction mathématique. Le notebook est conçu pour rester un bon exercice de coloration d'arêtes même si le résultat tombait : Petersen, les ponts et les échelles de Möbius ne doivent rien à un papier de 2026.

***
## 6. Exemples guidés et exercices

Trois exercices pour vous approprier le matériel. Les stubs s'exécutent tels quels (ils rendent `None`) : complétez-les, puis relancez la cellule.

Les trois premiers énoncés ont été résolus par un rendu de TP (PR étudiante #25629) : chaque résolution est conservée comme **exemple guidé crédité**, suivie d'un exercice neuf sur le même sujet.

#### Exemple guidé 1 — Indice chromatique générique (crédité #25629 — groupe 7 : Yasmine Nehad, Adam Benlaharche et Solène Martinez)

> Résolution originale : **groupe 7** (TP1, IA2-Min1, EPF), PR étudiante [#25629](https://github.com/jsboige/CoursIA/pull/25629). Code repris tel quel ; réponse resserrée.

Écrivez `indice_chromatique(G)` qui calcule $\chi'(G)$ en essayant successivement $k = \Delta(G)$ puis $k = \Delta(G) + 1$ (borne de Vizing). Vérifiez : $\chi'(K_4) = 3$, $\chi'(\text{Petersen}) = 4$, $\chi'(K_{3,3}) = 3$, $\chi'(\text{graphe à pont}) = 4$.

In [26]:
// Exercice 1 : indice chromatique generique
fn indice_chromatique(g: &Graphe) -> Option<usize> {
    let delta = g
        .adj
        .iter()
        .map(|vois| vois.len())
        .max()
        .unwrap_or(0);

    // Premier essai : k = delta (classe 1 ?)
    let (coloration, statut) = coloration_aretes_sat(g, delta, 10_000);
    if coloration.is_some() {
        return Some(delta);
    }
    if statut != "UNSAT" {
        // UNKNOWN : on ne peut rien conclure
        println!("statut indetermine pour k = {} : {}", delta, statut);
        return None;
    }

    // Deuxieme essai : k = delta + 1, toujours faisable d'apres Vizing
    let (coloration2, statut2) = coloration_aretes_sat(g, delta + 1, 10_000);
    assert!(
        coloration2.is_some(),
        "contredit Vizing ?! statut = {}",
        statut2
    );
    Some(delta + 1)
}

fn complet_biparti(p: usize, q: usize) -> Graphe {
    let noms = (0..p + q).map(|i| i.to_string()).collect();
    let mut g = Graphe::nouveau(noms);
    for i in 0..p {
        for j in p..p + q {
            g.ajoute(i, j);
        }
    }
    g
}

let K33 = complet_biparti(3, 3);
let cas: Vec<(&str, Graphe, usize)> = vec![
    ("K4", TEMOINS[0].1.clone(), 3),
    ("Petersen", TEMOINS[2].1.clone(), 4),
    ("K33", K33.clone(), 3),
    ("cubique a pont", TEMOINS[3].1.clone(), 4),
];
for (nom, g, attendu) in &cas {
    let chi = indice_chromatique(g)
        .expect("indice chromatique indetermine");
    println!(
        "{:<15} chi' = {}  (attendu {})  {}",
        nom,
        chi,
        attendu,
        if chi == *attendu { "OK" } else { "ERREUR" }
    );
    assert_eq!(chi, *attendu);
}

K4              chi' = 3  (attendu 3)  OK


Petersen        chi' = 4  (attendu 4)  OK


K33             chi' = 3  (attendu 3)  OK


cubique a pont  chi' = 4  (attendu 4)  OK


()

**Réponse (groupe 7, #25629).** Les quatre valeurs attendues sont confirmées : $K_4$ et $K_{3,3}$ sont de **classe 1** ($\chi' = \Delta = 3$), tandis que le Petersen et le cubique à pont sont de **classe 2** ($\chi' = \Delta + 1 = 4$) — pour deux raisons différentes : le premier est un snark (section 3), le second a un pont (section 4). Le théorème de Vizing garantit que deux appels au solveur suffisent toujours : si $k = \Delta$ est prouvé INFEASIBLE, alors $\chi' = \Delta + 1$ sans chercher plus loin. Et un statut UNKNOWN n'est jamais pris pour une impossibilité : la fonction rend `None`, car seul INFEASIBLE (UNSAT) certifie (section 3).

#### Exercice 1 : classes de Vizing des graphes complets

Le théorème de Vizing encadre $\chi'(G)$ entre $\Delta(G)$ et $\Delta(G)+1$ pour **tout** graphe — pas seulement les cubiques. Sur les graphes complets, la réponse exacte est connue :

$$\chi'(K_n) = \begin{cases} n - 1 & \text{si } n \text{ est pair} \\ n & \text{si } n \text{ est impair} \end{cases}$$

Écrivez `classes_vizing_complets(n_max)` qui, pour chaque $n$ de 3 à `n_max`, calcule $\chi'(K_n)$ avec `indice_chromatique`, le compare à la forme fermée ci-dessus, et renvoie la liste des $n$ tels que $K_n$ est de **classe 2**. Justifiez ensuite la forme fermée pour $n$ impair en une phrase de dénombrements : combien d'arêtes couvre au plus un couplage de $K_n$ ?

In [27]:
// Exercice 1 : classes de Vizing des graphes complets
fn classes_vizing_complets(_n_max: usize) -> Option<Vec<usize>> {
    // TODO: classez les graphes complets K_n par classe de Vizing
    // Etape 1 : construisez K_n avec graphe_complet(n) et appelez indice_chromatique
    // Etape 2 : comparez chi'(K_n) a la forme fermee (n - 1 si n pair, n sinon)
    // Etape 3 : renvoyez la liste des n tels que K_n est de classe 2
    // Indice : dans K_n avec n impair, un couplage couvre au plus (n - 1)/2 aretes,
    //          et K_n compte n(n - 1)/2 aretes -- bornez k par en-dessous
    println!("Exercice a completer");
    None
}

classes_vizing_complets(7);

// Tests attendus (decommentez apres implementation) :
// let classe2 = classes_vizing_complets(7).unwrap();
// println!("K_n de classe 2 pour n dans : {:?}", classe2);
// assert_eq!(classe2, vec![3, 5, 7]);

Exercice a completer


#### Exemple guidé 2 — Chasse au snark (crédité #25629 — groupe 7 : Yasmine Nehad, Adam Benlaharche et Solène Martinez)

> Résolution originale : **groupe 7** (TP1, IA2-Min1, EPF), PR étudiante [#25629](https://github.com/jsboige/CoursIA/pull/25629). Code repris tel quel ; réponse resserrée.

Écrivez `chasse_snark(n, essais)` qui échantillonne des cubiques aléatoires sans pont jusqu'à en trouver une non 3-colorable (un snark), et la renvoie — ou `None` après épuisement des essais. À $n = 10$, le seul snark possible est le Petersen : la chasse est donc un test de rareté. Combien d'essais faut-il ?

In [28]:
// Exercice 2 : chasse au snark
fn est_isomorphe(g1: &Graphe, g2: &Graphe) -> bool {
    // Backtracking : cherche une bijection sommets a sommets preservant l'adjacence
    let n = g1.n();
    if n != g2.n() || g1.m() != g2.m() {
        return false;
    }
    let adjacents = |g: &Graphe, a: usize, b: usize| g.adj[a].contains(&b);
    fn essaie(
        g1: &Graphe,
        g2: &Graphe,
        correspondance: &mut Vec<usize>,
        utilise: &mut Vec<bool>,
        adjacents: &dyn Fn(&Graphe, usize, usize) -> bool,
    ) -> bool {
        let n = g1.n();
        let i = correspondance.len();
        if i == n {
            return true;
        }
        for c in 0..n {
            if utilise[c] || g1.adj[i].len() != g2.adj[c].len() {
                continue;
            }
            // coherence : les images des sommets deja places doivent conserver l'adjacence
            let coherent = (0..i).all(|j| {
                adjacents(g1, i, j) == adjacents(g2, c, correspondance[j])
            });
            if !coherent {
                continue;
            }
            utilise[c] = true;
            correspondance.push(c);
            if essaie(g1, g2, correspondance, utilise, adjacents) {
                return true;
            }
            correspondance.pop();
            utilise[c] = false;
        }
        false
    }
    let mut correspondance = Vec::new();
    let mut utilise = vec![false; n];
    essaie(g1, g2, &mut correspondance, &mut utilise, &adjacents)
}

fn chasse_snark(n: usize, essais: usize, graine: u64) -> Option<(Graphe, usize)> {
    let mut rng = StdRng::seed_from_u64(graine);
    for i in 1..=essais {
        let graine_g = rng.next_u32() as u64;
        let mut rng_g = StdRng::seed_from_u64(graine_g);
        let g = graphe_regulier_aleatoire(3, n, &mut rng_g);
        if !sans_pont(&g) {
            continue;
        }
        let (coloration, statut) = coloration_aretes_sat(&g, 3, 10_000);
        if statut == "UNSAT" {
            return Some((g, i)); // on garde le nombre de tirages necessaires
        }
    }
    None
}

let snark = chasse_snark(10, 2000, SEED);
match &snark {
    Some((g, tirages)) => {
        println!("snark trouve : {}", v(true));
        println!("nombre de tirages : {}", tirages);
        println!("isomorphe a Petersen : {}", v(est_isomorphe(g, &PETERSEN)));
    }
    None => {
        println!("snark trouve : {}", v(false));
    }
}

// Combien d'essais en moyenne ? On relance la chasse avec plusieurs graines
let mut nb_essais: Vec<usize> = Vec::new();
for s in 0..10u64 {
    if let Some((g, tirages)) = chasse_snark(10, 5000, s) {
        assert!(est_isomorphe(&g, &PETERSEN));
        nb_essais.push(tirages);
    }
}
println!("essais par chasse (10 graines) : {:?}", nb_essais);
let moyenne: f64 = nb_essais.iter().map(|&x| x as f64).sum::<f64>() / nb_essais.len() as f64;
println!("moyenne : {:.1}", moyenne);

snark trouve : True


nombre de tirages : 902


isomorphe a Petersen : True


essais par chasse (10 graines) : [254, 2025, 17, 272, 13, 900, 1381, 684, 288, 78]


moyenne : 591.2


**Réponse (groupe 7, #25629).** La chasse aboutit, et le snark trouvé est toujours isomorphe au graphe de Petersen — c'est inévitable, puisque c'est l'unique snark à 10 sommets. Le nombre de tirages nécessaires varie fortement d'une graine à l'autre : ici de 13 à 2025, pour une moyenne de 591.2 sur 10 graines (le générateur aléatoire de Rust ne suit pas le même flux que celui de Python : les tirages diffèrent, le verdict est identique). Cette rareté s'explique par la grande symétrie du Petersen (120 automorphismes) : un graphe très symétrique n'admet que $10!/120 = 30\,240$ étiquelages distincts, une toute petite fraction des graphes cubiques étiquetés de cette taille. Les snarks sont des objets rares — c'est aussi pourquoi l'échantillonnage de la section 5 n'en a trouvé aucun.

#### Exercice 2 : le snark de Tietze, au-delà du Petersen

Le Petersen est le plus petit snark, mais on en construit de plus gros par une **extension de sommet** : retirez un sommet $v$ d'un cubique, ajoutez un triangle $x$-$y$-$z$, puis reconnectez les trois anciens voisins $a, b, c$ de $v$ par $x$–$a$, $y$–$b$, $z$–$c$. Appliquée au Petersen, cette opération produit le **graphe de Tietze** (12 sommets).

Écrivez `etendre_sommet_triangle(G, v)` qui réalise cette opération, construisez le Tietze, puis certifiez en une seule passe que c'est bien un snark au-delà du Petersen : cubique, sans pont, `INFEASIBLE` en $k = 3$, coloriable en $k = 4$, non planaire, **non apex**, et non isomorphe au Petersen. Ce graphe contredit-il le théorème de 2026 ? Justifiez en une phrase.

In [29]:
// Exercice 2 : le snark de Tietze, audela du Petersen
fn etendre_sommet_triangle(_g: &Graphe, _v: usize) -> Option<Graphe> {
    // TODO: etendez le sommet v de G en un triangle
    // Etape 1 : copiez G, relevez les trois voisins de v, supprimez v
    // Etape 2 : ajoutez le triangle x-y-z (trois nouveaux sommets)
    // Etape 3 : reliez chaque ancien voisin a un sommet distinct du triangle
    // Indice : un sommet du triangle a deja deux voisins internes, il lui faut
    //          exactement un voisin externe pour que le graphe reste cubique
    println!("Exercice a completer");
    None
}

etendre_sommet_triangle(&PETERSEN, 0);

// Tests attendus (decommentez apres implementation) :
// let tietze = etendre_sommet_triangle(&PETERSEN, 0).unwrap();
// let d = decrire("Tietze", &tietze);
// println!("{:?}", d);
// let (col3t, st3) = coloration_aretes_sat(&tietze, 3, 10_000);
// let (col4t, st4) = coloration_aretes_sat(&tietze, 4, 10_000);
// println!("k=3 : {} | k=4 : {}", st3, st4);
// assert!(d.cubique && d.sans_pont && !d.planaire && !d.apex);
// assert!(st3 == "UNSAT" && col4t.is_some());
// assert!(!est_isomorphe(&tietze, &PETERSEN));

Exercice a completer


#### Exemple guidé 3 — Échelles de Möbius plus grandes (crédité #25629 — groupe 7 : Yasmine Nehad, Adam Benlaharche et Solène Martinez)

> Résolution originale : **groupe 7** (TP1, IA2-Min1, EPF), PR étudiante [#25629](https://github.com/jsboige/CoursIA/pull/25629). Code repris tel quel ; réponse conservée.

Les $M_8$, $M_{10}$, $M_{12}$ sont apex et 3-colorables. Cela persiste-t-il en grandissant ? Écrivez `explorer_mobius(tailles)` qui renvoie, pour chaque $n$ pair, un dictionnaire `{n: {"planaire": ..., "apex": ..., "3-colorable": ...}}`. Y a-t-il une taille où l'apex-ité disparaît — et si oui, la 3-coloration survit-elle (les échelles de Möbius sont-elles des snarks pour certains $n$) ?

In [30]:
// Exercice 3 : echelles de Mobius en grand
fn explorer_mobius(tailles: &[usize]) -> Vec<(usize, bool, bool, bool, String)> {
    let mut resultats = Vec::new();
    for &n in tailles {
        if n % 2 != 0 {
            continue; // M_n n'est defini que pour n pair
        }
        let g = echelle_mobius(n);
        let d = decrire(&format!("M{}", n), &g);
        let (coloration, statut) = coloration_aretes_sat(&g, 3, 10_000);
        resultats.push((n, d.planaire, d.apex, coloration.is_some(), statut));
    }
    resultats
}

let df_grand = explorer_mobius(&[14, 16, 18, 20, 30, 40]);
println!("{:>4}  {:>9}  {:>5}  {:>11}  statut", "n", "planaire", "apex", "3-colorable");
for (n, pl, ap, c3, statut) in &df_grand {
    println!(
        "{:>4}  {:>9}  {:>5}  {:>11}  {}",
        n,
        v(*pl),
        v(*ap),
        v(*c3),
        statut
    );
}

   n   planaire   apex  3-colorable  statut


  14      False   True         True  SAT


  16      False   True         True  SAT


  18      False   True         True  SAT


  20      False   True         True  SAT


  30      False   True         True  SAT


  40      False   True         True  SAT


()

**Réponse (groupe 7, #25629).** Aucune transition n'apparaît : pour toutes les tailles testées ($n$ de 14 à 40), $M_n$ reste non planaire, apex et 3-colorable (statut `SAT`). Ce n'est pas un hasard, et cela se démontre pour tout $n$ pair :

- **apex** : une échelle de Möbius se dessine avec un seul croisement (le « vrillage » du ruban). En supprimant une extrémité de l'une des deux arêtes qui se croisent, on retire ce croisement et le graphe restant est planaire.
- **3-colorable** : le cycle extérieur a $n$ arêtes avec $n$ pair, donc il se colorie en alternant deux couleurs ; les barreaux forment un couplage parfait et prennent tous la troisième couleur.

Les échelles de Möbius ne sont donc jamais des snarks. Elles illustrent le théorème de 2026 : des graphes non planaires, mais apex et 3-colorables pour toutes les tailles.

#### Exercice 3 : la réciproque du théorème apex est-elle vraie ?

Le théorème de 2026 est une **implication** : cubique sans pont **et apex** $\Rightarrow$ 3-colorable. La réciproque est-elle vraie — tout cubique sans pont 3-colorable est-il apex ? La section 5 a montré que la zone apex-non-planaire est habitable ; cherchons maintenant un cubique sans pont, **non apex**, qui reste 3-colorable.

Écrivez `classer_famille(graphes)` qui renvoie, pour chaque graphe nommé, `{"planaire": ..., "apex": ..., "3-colorable": ..., "statut": ...}` à partir de `decrire` et de `coloration_aretes_sat(k=3)`, puis affichez la table pour la famille ci-dessous. Répondez : quels graphes de la famille sont non-apex et pourtant 3-colorables, et pourquoi cela ne contredit-il pas le théorème ?

In [31]:
// Exercice 3 : la reciproque du theoreme apex est-elle vraie ?
fn classer_famille(
    _graphes: &[(String, Graphe)],
) -> Option<Vec<(String, bool, bool, bool, String)>> {
    // TODO: classez chaque graphe de la famille (planaire / apex / 3-colorable)
    // Etape 1 : appelez decrire(nom, g) pour les proprietes structurelles
    // Etape 2 : appelez coloration_aretes_sat(g, 3, 10_000) et relevez le statut
    // Etape 3 : rassemblez dans Vec<(nom, planaire, apex, 3-colorable, statut)>
    //           puis identifiez les non-apex qui restent 3-colorables
    // Indice : un tel graphe ne met PAS le theoreme en defaut -- l'implication ne
    //          va que dans un sens ; pensez au graphe de Heawood
    println!("Exercice a completer");
    None
}

classer_famille(&[]);

// Constructeurs pour la famille attendue (decommentez apres implementation) :
fn hypercube3() -> Graphe {
    let noms = (0..8).map(|i| format!("{:03b}", i)).collect();
    let mut g = Graphe::nouveau(noms);
    for u in 0..8 {
        for b in 0..3 {
            g.ajoute(u, u ^ (1 << b));
        }
    }
    g
}

fn heawood() -> Graphe {
    // Graphe de Heawood : 7 points, 7 droites de l'incidence de Fano ;
    // la droite i porte les points i, i+1 et i+3 (mod 7).
    let mut noms: Vec<String> = (0..7).map(|i| format!("p{}", i)).collect();
    for i in 0..7 {
        noms.push(format!("d{}", i));
    }
    let mut g = Graphe::nouveau(noms);
    for i in 0..7 {
        g.ajoute(i, 7 + i);
        g.ajoute(i, 7 + (i + 1) % 7);
        g.ajoute(i, 7 + (i + 3) % 7);
    }
    g
}

// Tests attendus (decommentez apres implementation) :
// let famille: Vec<(String, Graphe)> = vec![
//     ("K33".to_string(), complet_biparti(3, 3)),
//     ("cube Q3".to_string(), hypercube3()),
//     ("M8".to_string(), echelle_mobius(8)),
//     ("Petersen".to_string(), PETERSEN.clone()),
//     ("Heawood".to_string(), heawood()),
// ];
// let classement = classer_famille(&famille).unwrap();
// for (nom, pl, ap, c3, statut) in &classement {
//     println!("{:<10} planaire={} apex={} 3-colorable={} statut={}", nom, v(*pl), v(*ap), v(*c3), statut);
// }

Exercice a completer


***
## 7. Conclusion et perspectives

**Ce que nous avons établi** (vérification computationnelle, pas preuve) :

1. La 3-coloration d'arêtes d'un cubique se formule comme un CSP SAT où chaque couleur est un **couplage parfait** — le solveur rend des témoins (SAT) comme des certificats d'impossibilité (UNSAT).
2. Le **Petersen** est le snark minimal : $\chi' = 4 = \Delta + 1$, non planaire et non apex.
3. Un **pont** rend la 3-coloration impossible par un argument de parité en trois lignes — le solveur le constate, la mathématique l'explique.
4. Les **échelles de Möbius** habitent la zone apex-non-planaire couverte par le théorème de 2026 et pas par Tutte-1966 ; toutes les instances apex testées sont 3-colorables, en accord avec le théorème.

**Perspectives** : un compagnon Lean (tranche B de l'issue #13031) formalisera les définitions (cubique, sans pont, apex, 3-arête-colorable) sur `Mathlib.Combinatorics.SimpleGraph` et vérifiera exécutivement que le Petersen n'est pas 3-arête-colorable. La formalisation de la preuve complète (réductibilité + déchargement) est explicitement hors périmètre : c'est l'échelle du théorème des quatre couleurs (années-homme, cf. Gonthier).

**Références** :
- [Version Python de ce carnet](Frontieres-01-EdgeColoring-Tutte-Python.ipynb) — le même parcours avec CP-SAT et networkx
- [App-2 : Coloration de Graphes](../Part3-Cas-pratiques/App-02-GraphColoring-Python.ipynb) — coloration de sommets
- Vizing (1964), *On an estimate of the chromatic class of a p-graph*
- Tutte (1966), *On the algebraic characterization of some graph classes*
- arXiv 2608.22870 (2026), *Three-edge-coloring apex cubic graphs* — preuve assistée par ordinateur, algorithme $O(n^2)$
- Robertson, Sanders, Seymour, Thomas (1997), *The four-colour theorem* — le cadre historique des preuves par ordinateur